In [1]:
2+2

4

In [13]:
import pandas as pd

# Define the exact file path on your E: drive
file_path = r"E:\Pakistan_Unified_Tourism_Master_Dataset.xlsx"

# 1. Load the COMPLETE First Sheet (Destination Metrics)
# skips the decorative title block to grab the table headers perfectly
df_metrics = pd.read_excel(file_path, sheet_name="Destination Metrics", skiprows=4)

# 2. Load the COMPLETE Second Sheet (Reviews & Accommodations)
df_reviews = pd.read_excel(file_path, sheet_name="Reviews & Accommodations", skiprows=4)

# ---- Complete Data Cleaning ----
# Drop the empty structural padding column if captured by the parser
if 'Unnamed: 0' in df_metrics.columns:
    df_metrics = df_metrics.drop(columns=['Unnamed: 0'])
if 'Unnamed: 0' in df_reviews.columns:
    df_reviews = df_reviews.drop(columns=['Unnamed: 0'])

# Filter out the final structural row ('Average Metric') to ensure clean data types
df_metrics = df_metrics[df_metrics['Destination'].str.contains("Average Metric", na=False) == False]

# Reset the indexes so it counts cleanly from 0 to 34
df_metrics = df_metrics.reset_index(drop=True)
df_reviews = df_reviews.reset_index(drop=True)

# ---- Force Python to Display the Entire Dataset in Console ----
pd.set_option('display.max_rows', None)      # Tells pandas never to truncate rows
pd.set_option('display.max_columns', None)   # Tells pandas never to truncate columns
pd.set_option('display.width', 1000)         # Prevents rows from wrapping awkwardly
pd.set_option('display.max_colwidth', None)  # Shows complete unedited review strings

# ---- Print the Entire Tables ----
print("==========================================================================")
print("             TAB 1: COMPLETE DESTINATION METRICS DATASET                   ")
print("==========================================================================")
print(df_metrics)

print("\n\n==========================================================================")
print("           TAB 2: COMPLETE REVIEWS & ACCOMMODATIONS DATASET               ")
print("==========================================================================")
print(df_reviews)

             TAB 1: COMPLETE DESTINATION METRICS DATASET                   
              Destination    District       Province / Region  Latitude  Longitude Budget Index Weather Profile Trip Style Peak Month Crowd Profile  Satisfaction (%)
0                   Hunza       Hunza        Gilgit-Baltistan   36.3167    74.6500         High            Cold  Adventure   December        Medium              92.0
1                  Skardu      Skardu        Gilgit-Baltistan   35.2974    75.6333       Medium            Cold  Adventure       July          High              89.0
2                  Murree      Murree                  Punjab   33.9070    73.3943          Low            Cold     Family    January          High              75.0
3                    Swat        Swat      Khyber Pakhtunkhwa   34.8065    72.3548       Medium            Mild     Family      March        Medium              85.0
4                   Naran    Mansehra      Khyber Pakhtunkhwa   34.9056    73.6517       Mediu

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import joblib

# Load dataset
file_path = r"E:\Pakistan_Unified_Tourism_Master_Dataset.xlsx"
df = pd.read_excel(file_path, sheet_name="Destination Metrics", skiprows=4)

# Clean
# Clean dataset properly
df = df.drop(columns=['Unnamed: 0'], errors='ignore')

df = df[df['Destination'].notna()]

df.columns = df.columns.str.strip()

# 🔥 IMPORTANT FIX: remove ALL missing values
df = df.dropna()

# Convert Satisfaction safely
df['Satisfaction (%)'] = pd.to_numeric(df['Satisfaction (%)'], errors='coerce')

# Drop again after conversion
df = df.dropna()

# FEATURES (REALISTIC PREDICTORS)
features = [
    'Budget Index',
    'Weather Profile',
    'Trip Style',
    'Peak Month',
    'Crowd Profile'
]

# TARGET (VERY IMPORTANT CHANGE)
target = 'Satisfaction (%)'

X = df[features]
y = df[target]

# Preprocessing
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), features)
])

# Model
model = Pipeline([
    ('preprocess', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=300,
        random_state=42
    ))
])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Score
r2 = r2_score(y_test, y_pred)

print("🎯 R2 Score:", round(r2, 3))

# Save
joblib.dump(model, "tourism_satisfaction_model.pkl")

print("✅ MODEL TRAINED SUCCESSFULLY")
import joblib
import pandas as pd

# Load your dataset (same as you already did)
df = pd.read_excel(r"E:\Pakistan_Unified_Tourism_Master_Dataset.xlsx", sheet_name="Destination Metrics", skiprows=4)

# Clean
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
df = df[df['Destination'].str.contains("Average Metric", na=False) == False]

# Features used in ML
X = df[['Budget Index','Weather Profile','Trip Style','Peak Month']]

# One-hot encode (THIS is the key step)
X_encoded = pd.get_dummies(X)

# SAVE THE REAL COLUMNS
joblib.dump(X_encoded.columns, "model_columns.pkl")

print("Saved columns:", list(X_encoded.columns))

🎯 R2 Score: 0.542
✅ MODEL TRAINED SUCCESSFULLY
Saved columns: ['Budget Index_High', 'Budget Index_Low', 'Budget Index_Medium', 'Weather Profile_Cold', 'Weather Profile_Hot', 'Weather Profile_Mild', 'Trip Style_Adventure', 'Trip Style_Beach', 'Trip Style_City', 'Trip Style_Cultural', 'Trip Style_Family', 'Peak Month_April', 'Peak Month_August', 'Peak Month_December', 'Peak Month_February', 'Peak Month_January', 'Peak Month_July', 'Peak Month_June', 'Peak Month_March', 'Peak Month_May', 'Peak Month_November', 'Peak Month_October', 'Peak Month_September']


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score
from tensorflow import keras
from tensorflow.keras import layers

# Load data
file_path = r"E:\Pakistan_Unified_Tourism_Master_Dataset.xlsx"
df = pd.read_excel(file_path, sheet_name="Destination Metrics", skiprows=4)

# Clean
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
df = df.dropna()
df.columns = df.columns.str.strip()

# Features
features = [
    'Budget Index',
    'Weather Profile',
    'Trip Style',
    'Peak Month',
    'Crowd Profile'
]

target = 'Satisfaction (%)'

X = df[features]
y = df[target]

# PREPROCESSING FIX (IMPORTANT)
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), features)
])

X = preprocessor.fit_transform(X)

# SCALE TARGET (IMPORTANT FOR NN STABILITY)
y = StandardScaler().fit_transform(y.values.reshape(-1, 1))

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Convert to dense for TensorFlow
X_train = X_train.toarray()
X_test = X_test.toarray()

# Neural Network (improved)
model = keras.Sequential([
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

# Train (IMPORTANT: shuffle=True helps a LOT)
model.fit(
    X_train,
    y_train,
    epochs=120,
    batch_size=4,
    verbose=1,
    shuffle=True
)

# Predict
y_pred = model.predict(X_test)

# Reverse scaling
y_test = StandardScaler().fit(df[target].values.reshape(-1,1)).transform(y_test)
y_pred = StandardScaler().fit(df[target].values.reshape(-1,1)).transform(y_pred)

# R2 Score
r2 = r2_score(y_test, y_pred)

print("\n🔥 FIXED Deep Learning R2:", round(r2, 3))

import joblib
joblib.dump(preprocessor, "dl_preprocessor.pkl")
model.save("satisfaction_model.h5")

Epoch 1/120
7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - loss: 1.0030 - mae: 0.7555
Epoch 2/120
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.8078 - mae: 0.6593
Epoch 3/120
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.7063 - mae: 0.6059 
Epoch 4/120
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.5754 - mae: 0.5271
Epoch 5/120
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.4561 - mae: 0.4530
Epoch 6/120
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.3579 - mae: 0.4164 
Epoch 7/120
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.2770 - mae: 0.3728 
Epoch 8/120
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1846 - mae: 0.3088 
Epoch 9/120
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.1888 - mae: 0.2986
Epoch 10/120
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.1674 - mae: 0.2977  
Epoch 11/120
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.1532 - mae: 0.2840
Epoch 12/120
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0868 - mae: 0.2095
Epoch 13/120
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms


🔥 FIXED Deep Learning R2: 0.679


In [36]:
import joblib

# -----------------------------
# SAVE TRAINED MODEL
# -----------------------------
joblib.dump(model, "tourism_model.pkl")

# -----------------------------
# SAVE PREPROCESSOR (IMPORTANT)
# -----------------------------
joblib.dump(preprocessor, "preprocessor.pkl")

# -----------------------------
# SAVE FEATURE NAMES (FOR STREAMLIT ALIGNMENT)
# -----------------------------
feature_names = preprocessor.get_feature_names_out()
joblib.dump(feature_names, "model_columns.pkl")

# -----------------------------
# SAVE TARGET ENCODER (IF USED)
# -----------------------------
from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(df[target])

globals()['target_encoder'] = target_encoder
if 'target_encoder' in globals():
    joblib.dump(target_encoder, "target_encoder.pkl")
# Save sentiment model (ONLY IF TRAINED)
if 'sentiment_model' in globals():
    sentiment_model_obj = globals().get("sentiment_model")
    if sentiment_model_obj is not None:
        joblib.dump(sentiment_model_obj, "sentiment_model.pkl")


print("✅ All models saved cleanly!")

✅ All models saved cleanly!


In [11]:
import streamlit as st
import joblib
import pandas as pd
# STREAMLIT UI
# -----------------------------
st.title("🌍 Smart Tourism AI System")
st.write("AI-powered travel recommendation + prediction + sentiment analysis")

# -----------------------------
# USER INPUTS
# -----------------------------
budget = st.selectbox("Budget Index", ["Low", "Medium", "High"])
weather = st.selectbox("Weather Profile", ["Cold", "Mild", "Hot"])
trip = st.selectbox("Trip Style", ["Adventure", "Family", "Cultural", "Beach", "City"])
month = st.selectbox("Peak Month", 
    ["January","February","March","April","May","June",
     "July","August","September","October","November","December"])
crowd = st.selectbox("Crowd Profile", ["Low", "Medium", "High"])

# -----------------------------
# DESTINATION PREDICTION (ML)
# -----------------------------
if st.button("🌍 Recommend Destination"):

    input_df = pd.DataFrame([[budget, weather, trip, month]],
                            columns=["Budget Index","Weather Profile","Trip Style","Peak Month"])

    input_encoded = pd.get_dummies(input_df)
    # Load the saved columns from your trained model
    columns = joblib.load('model_columns.pkl')
    input_encoded = input_encoded.reindex(columns=columns, fill_value=0)

    pred = rf_model.predict(input_encoded)
    destination = encoder.inverse_transform(pred)

    st.success(f"Recommended Destination: {destination[0]}")

# -----------------------------
# DEEP LEARNING: SATISFACTION
# -----------------------------
if st.button("⭐ Predict Satisfaction"):

    dl_input = pd.DataFrame([[budget, weather, trip, month, crowd]],
                            columns=["Budget Index","Weather Profile","Trip Style","Peak Month","Crowd Profile"])

    # USE TRAINED PREPROCESSOR (CRITICAL FIX)
    dl_input_processed = dl_preprocessor.transform(dl_input)

    prediction = dl_model.predict(dl_input_processed)[0][0]

    st.info(f"Predicted Satisfaction Score: {round(float(prediction),2)} %")

# -----------------------------
# SENTIMENT ANALYSIS
# -----------------------------
st.subheader("💬 Tourist Review Sentiment Analysis")

review = st.text_area("Enter a tourist review")

if st.button("Analyze Sentiment"):
    result = sentiment_model.predict([review])[0]
    st.success(f"Sentiment: {result}")

# -----------------------------
# FOOTER
# -----------------------------
st.write("---")
st.write("🚀 Built by Smart Tourism AI System (ML + DL Project)")

2026-05-25 22:48:06.987 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-25 22:48:06.989 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-25 22:48:06.996 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-25 22:48:06.999 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-25 22:48:07.001 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-25 22:48:07.002 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-25 22:48:07.004 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-25 22:48:07.006 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar